## Llamando las librerías que usaré

In [1]:
import XRootD
import copy # copiar variables
import os   # gestionar rutas
import uproot   # Leer archivos ROOT sin usar ROOT directamente.
import awkward as ak  # Manejar arreglos tipo "jagged" (listas de listas, una por evento).
import numpy as np
import pandas as pd
import subprocess
import gc

## Estoy usando los datos de: 

## https://opendata.cern.ch/record/80036

In [2]:
# Leer el .txt
with open("archivos_ATLAS.txt") as f:
    archivos = [
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith("#")
    ]

# Tomar el primer archivo
archivo = archivos[0]

print("Archivo usado:", archivo)

Archivo usado: root://eospublic.cern.ch//eos/opendata/atlas/rucio/data15_hi/DAOD_HION14.41691915._000001.pool.root.1


## Revisando que ramas hay en un archivo .ROOT

In [3]:
print("\nObjetos dentro del ROOT:")
for key in uproot.open(archivo).keys():
    print("-", key)


Objetos dentro del ROOT:
- ##Params;2
- ##Params;1
- ##Shapes;2
- ##Shapes;1
- ##Links;2
- ##Links;1
- CollectionTree;1
- POOLContainer;1
- POOLContainerForm;1
- POOLCollectionTree;1
- MetaData;1
- MetaDataHdr;1
- MetaDataHdrForm;1


### El árbol "CollectionTree" tiene la información de las ramas que necesito

## Leyendo una cantidad grande de archivos para tener suficiente estadística y almacenando esa información en variables que usaré posteriormente.

In [ ]:
# =========================
# 0) Ajustes XRootD
# =========================
os.environ["XRD_REQUESTTIMEOUT"]   = "300"
os.environ["XRD_TIMEOUT"]          = "300"
os.environ["XRD_CONNECTIONRETRY"]  = "20"
os.environ["XRD_CONNECTIONWINDOW"] = "120"

TREE_NAME = "CollectionTree"

# =========================
# 1) Config
# =========================
N_OK   = 100
N_TRY  = 621
BLOQUE = 1

archivos_try = archivos[:min(N_TRY, len(archivos))]

OUTDIR = "parquets_final"
os.makedirs(OUTDIR, exist_ok=True)

# =========================
# 2) Ramas necesarias
# =========================
BR_QOVERP        = "InDetTrackParticlesAuxDyn.qOverP"
BR_THETA         = "InDetTrackParticlesAuxDyn.theta"
BR_PHI           = "InDetTrackParticlesAuxDyn.phi"
BR_CENTRALITYMIN = "EventInfoAuxDyn.CentralityMin"
BR_CENTRALITYMAX = "EventInfoAuxDyn.CentralityMax"
BR_RUN           = "EventInfoAuxDyn.runNumber"
BR_EVT           = "EventInfoAuxDyn.eventNumber"

# ramas obligatorias
ramas_base = [
    BR_QOVERP,
    BR_THETA,
    BR_PHI,
    BR_CENTRALITYMIN,
    BR_CENTRALITYMAX,
    BR_RUN,
    BR_EVT,
]

# ramas opcionales
ramas_opcionales = []

ramas_all = ramas_base + ramas_opcionales

# =========================
# 3) Funciones auxiliares
# =========================
def split_root_url(u: str):
    assert u.startswith("root://")
    rest = u[len("root://"):]
    if "//" not in rest:
        raise ValueError(f"URL no tiene //path: {u}")
    host, path = rest.split("//", 1)
    return host, "/" + path

def xrdfs_stat_ok(u: str, timeout_s=15) -> bool:
    host, path = split_root_url(u)
    r = subprocess.run(
        ["xrdfs", f"root://{host}", "stat", path],
        capture_output=True,
        text=True,
        timeout=timeout_s
    )
    return (r.returncode == 0)

def concat_or_fail(lst, name):
    if len(lst) == 0:
        raise RuntimeError(f"No se juntó ningún chunk para {name}.")
    return ak.concatenate(lst, axis=0)

def nan_like_jagged(template, dtype=np.float32):
    """
    Devuelve un arreglo jagged con la misma estructura que 'template'
    pero rellenado con NaN.
    """
    return ak.values_astype(ak.full_like(template, np.nan), dtype)

# =========================
# 4) Procesar un bloque y guardarlo
# =========================
def procesar_y_guardar_bloque(archivos_bloque, bloque_id):
    chunks = {br: [] for br in ramas_all}
    read_ok_local = []
    read_fail_local = []
    stat_fail_local = []

    print(f"\n=========================")
    print(f"Procesando bloque {bloque_id:03d}")
    print(f"Archivos en este bloque: {len(archivos_bloque)}")
    print(f"=========================")

    for u in archivos_bloque:
        try:
            if not xrdfs_stat_ok(u, timeout_s=15):
                stat_fail_local.append(u)
                print("STAT FAIL:", u.split("_")[-1])
                continue
        except subprocess.TimeoutExpired:
            stat_fail_local.append(u)
            print("STAT TO  :", u.split("_")[-1])
            continue
        except Exception as e:
            stat_fail_local.append(u)
            print("STAT ERR :", u.split("_")[-1], "->", str(e).splitlines()[0])
            continue

        try:
            # Primero revisar qué ramas existen realmente
            with uproot.open(u, timeout=120) as f:
                tree = f[TREE_NAME]
                tree_keys = set(tree.keys())

            ramas_faltantes_base = [br for br in ramas_base if br not in tree_keys]
            if len(ramas_faltantes_base) > 0:
                raise RuntimeError(
                    "Faltan ramas obligatorias: " + ", ".join(ramas_faltantes_base)
                )

            ramas_presentes = ramas_base + [br for br in ramas_opcionales if br in tree_keys]

            for arrays in uproot.iterate(
                {u: TREE_NAME},
                ramas_presentes,
                step_size="2 MB",
                library="ak",
                how=dict,
                allow_missing=True,
            ):
                # obligatorias
                for br in ramas_base:
                    chunks[br].append(arrays[br])

                # opcionales
                # no hay ramas opcionales en este caso

            read_ok_local.append(u)
            print(f"OK iterate: {u.split('_')[-1]} | OK en bloque = {len(read_ok_local)}")

        except Exception as e:
            read_fail_local.append((u, str(e).splitlines()[0]))
            print("FAIL iterate:", u.split("_")[-1], "->", str(e).splitlines()[0])

    if len(read_ok_local) == 0:
        print(f"\nBloque {bloque_id:03d}: no se pudo leer ningún archivo.")
        return read_ok_local, read_fail_local, stat_fail_local

    qoverp_evt = concat_or_fail(chunks[BR_QOVERP], BR_QOVERP)
    theta_evt  = concat_or_fail(chunks[BR_THETA], BR_THETA)
    phi_evt    = concat_or_fail(chunks[BR_PHI], BR_PHI)

    centmin_evt = concat_or_fail(chunks[BR_CENTRALITYMIN], BR_CENTRALITYMIN)
    centmax_evt = concat_or_fail(chunks[BR_CENTRALITYMAX], BR_CENTRALITYMAX)

    runNumber_evt   = concat_or_fail(chunks[BR_RUN], BR_RUN)
    eventNumber_evt = concat_or_fail(chunks[BR_EVT], BR_EVT)

    # =========================
    # Seudorapidez a partir de theta
    # =========================
    eta_evt = -np.log(np.tan(theta_evt / 2.0))

    n_trk_por_evento = ak.to_numpy(ak.num(qoverp_evt, axis=1))
    run_rep = np.repeat(ak.to_numpy(runNumber_evt), n_trk_por_evento)
    evt_rep = np.repeat(ak.to_numpy(eventNumber_evt), n_trk_por_evento)

    centmin_rep = np.repeat(ak.to_numpy(centmin_evt), n_trk_por_evento)
    centmax_rep = np.repeat(ak.to_numpy(centmax_evt), n_trk_por_evento)

    qoverp_flat = ak.to_numpy(ak.flatten(qoverp_evt, axis=None))
    theta_flat  = ak.to_numpy(ak.flatten(theta_evt, axis=None))
    phi_flat    = ak.to_numpy(ak.flatten(phi_evt, axis=None))
    eta_flat    = ak.to_numpy(ak.flatten(eta_evt, axis=None))

    print("\nChequeo de tamaños:")
    print("run_rep      :", len(run_rep))
    print("evt_rep      :", len(evt_rep))
    print("centmin_rep  :", len(centmin_rep))
    print("centmax_rep  :", len(centmax_rep))
    print("qoverp_flat  :", len(qoverp_flat))
    print("theta_flat   :", len(theta_flat))
    print("phi_flat     :", len(phi_flat))
    print("eta_flat     :", len(eta_flat))

    n = len(qoverp_flat)
    assert len(run_rep) == n
    assert len(evt_rep) == n
    assert len(centmin_rep) == n
    assert len(centmax_rep) == n
    assert len(theta_flat) == n
    assert len(phi_flat) == n
    assert len(eta_flat) == n

    df_tracks_evt = pd.DataFrame({
        "runNumber": run_rep,
        "eventNumber": evt_rep,
        "CentralityMin": centmin_rep,
        "CentralityMax": centmax_rep,
        "qOverP": qoverp_flat,
        "theta": theta_flat,
        "phi": phi_flat,
        "eta": eta_flat,
    })

    print("\nTabla del bloque:")
    print(df_tracks_evt.head())
    print("Shape:", df_tracks_evt.shape)

    nombre_salida = os.path.join(OUTDIR, f"tracks_RAA_bloque_{bloque_id:03d}.parquet")
    df_tracks_evt.to_parquet(
        nombre_salida,
        index=False,
        compression="snappy"
    )

    print(f"\nArchivo guardado: {nombre_salida}")

    del chunks
    del qoverp_evt, theta_evt, phi_evt, eta_evt
    del centmin_evt, centmax_evt
    del runNumber_evt, eventNumber_evt
    del n_trk_por_evento, run_rep, evt_rep
    del centmin_rep, centmax_rep
    del qoverp_flat, theta_flat, phi_flat, eta_flat
    del df_tracks_evt
    gc.collect()

    return read_ok_local, read_fail_local, stat_fail_local

# =========================
# 5) Ejecutar por bloques
# =========================
read_ok_total = []
read_fail_total = []
stat_fail_total = []

bloque_id = 1
i = 0

print(f"Buscando {N_OK} archivos realmente leídos, en bloques de {BLOQUE} archivos...")

while i < len(archivos_try) and len(read_ok_total) < N_OK:
    archivos_bloque = archivos_try[i:i + BLOQUE]

    read_ok_local, read_fail_local, stat_fail_local = procesar_y_guardar_bloque(
        archivos_bloque,
        bloque_id
    )

    espacio = N_OK - len(read_ok_total)
    read_ok_total.extend(read_ok_local[:espacio])

    read_fail_total.extend(read_fail_local)
    stat_fail_total.extend(stat_fail_local)

    print("\nAcumulado hasta ahora:")
    print("OK iterate:", len(read_ok_total))
    print("FAIL iterate:", len(read_fail_total))
    print("FAIL/TO stat:", len(stat_fail_total))

    i += BLOQUE
    bloque_id += 1

print("\n=========================")
print("Resumen final")
print("=========================")
print("Objetivo N_OK:", N_OK)
print("OK iterate (leídos):", len(read_ok_total))
print("FAIL iterate:", len(read_fail_total))
print("FAIL/TO stat:", len(stat_fail_total))
print("Parquets guardados en:", OUTDIR)

Buscando 100 archivos realmente leídos, en bloques de 1 archivos...

Procesando bloque 001
Archivos en este bloque: 1
OK iterate: 000001.pool.root.1 | OK en bloque = 1

Chequeo de tamaños:
run_rep      : 7454243
evt_rep      : 7454243
centmin_rep  : 7454243
centmax_rep  : 7454243
qoverp_flat  : 7454243
theta_flat   : 7454243
phi_flat     : 7454243
eta_flat     : 7454243

Tabla del bloque:
   runNumber  eventNumber  CentralityMin  CentralityMax    qOverP     theta  \
0     286854      4187577           24.0           25.0  0.000223  2.901227   
1     286854      4187577           24.0           25.0  0.000098  2.941496   
2     286854      4187577           24.0           25.0  0.000281  2.774147   
3     286854      4187577           24.0           25.0 -0.000109  2.937641   
4     286854      4187577           24.0           25.0 -0.000273  2.772569   

        phi       eta  
0  0.609461 -2.113911  
1 -1.227039 -2.298758  
2 -0.816671 -1.682985  
3  1.723667 -2.279544  
4  1.515703 -

## Verificar si el .parquet se creó